In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

In [2]:
import os
import qnas_config as cfg
from util import check_files
from cnn.input import GenericDataLoader
from cnn.train_detailed import train_and_eval

In [3]:
phase = 'retrain'
experiment_path = os.path.join("experiments", "exp13_adamw_repeat_1")
config_file = 'config_files/config11.txt'

args = {
    'experiment_path': experiment_path,
    'config_file': config_file,
    'retrain_folder': 'retrain',
    'data_path': 'cifar10_data',
    'log_level': 'INFO',
    'max_epochs': 2,
    'epochs_to_eval': 10,
    'batch_size': 256,
    'eval_batch_size': 1000,
    'limit_data': False,
    'num_workers': 4,
    'device': 'cuda:1',
}

In [4]:
check_files(args['experiment_path'])
config = cfg.ConfigParameters(args, phase=phase)
config.get_parameters()

fn_dict=config.fn_dict

In [5]:
config.load_evolved_data(experiment_path=experiment_path)
params = config.train_spec
params

{'available_gpus': [0, 2],
 'batch_size': 256,
 'data_augmentation': True,
 'data_path': 'cifar10_data',
 'dataset': 'Cifar10',
 'decay': 0.9,
 'device': 'cuda:1',
 'epochs_to_eval': 10,
 'eval_batch_size': 1000,
 'experiment_path': 'experiments/exp13_adamw_repeat_1/retrain',
 'learning_rate': 0.001,
 'limit_data': False,
 'limit_data_value': 10000,
 'log_level': 'INFO',
 'max_epochs': 300,
 'mixed_precision': True,
 'momentum': 0.0,
 'num_workers': 4,
 'optimizer': 'AdamW',
 'phase': 'retrain',
 'save_checkpoints_epochs': 10,
 'save_summary_epochs': 0.25,
 'subtract_mean': True,
 'threads': 0,
 'weight_decay': 0.0001,
 'config_file': 'config_files/config11.txt',
 'retrain_folder': 'retrain'}

In [6]:
evolved_params = config.evolved_params
evolved_params['net']

['cbamdconv_3_1_256',
 'conv_3_1_32',
 'no_op',
 'conv_3_1_128',
 'no_op',
 'no_op',
 'conv_5_1_128',
 'no_op',
 'conv_3_1_32',
 'cbamdconv_3_1_64',
 'no_op',
 'max_pool_2_2',
 'conv_5_1_128',
 'max_pool_2_2',
 'no_op',
 'conv_5_1_64',
 'avg_pool_2_2',
 'avg_pool_2_2',
 'conv_5_1_32',
 'cbamdconv_1_1_128']

In [7]:
data_loader = GenericDataLoader(params=params)

In [8]:
train_loader, val_loader = data_loader.get_loader(pin_memory_device='cuda:1')
test_loader = data_loader.get_loader(for_train=False, pin_memory_device='cuda:1')

In [ ]:
retrain_multi = []
for i in range(1):
    result = train_and_eval(params=params, fn_dict=fn_dict, net_list=evolved_params['net'],
                            train_loader=train_loader, val_loader=val_loader, test_loader=test_loader)
    retrain_multi.append(result)
result

In [ ]:
with open(os.path.join(experiment_path, 'retrain_results.txt'), 'w') as f:
    for item in retrain_multi:
        f.write("%s\n" % item)